# Лабораторна робота № 2
## Прийняття рішень в умовах невизначеності

### Мета роботи
Навчитися приймати оптимальні рішення в умовах невизначеності, використовуючи методи та принципи теорії прийняття рішень.

### Опис прикладної області та задачі

Прикладна область — рослинництво. Фермерське господарство на півдні України вирішує, яку культуру посіяти наступного сезону на полі площею 100 га.

Урожайність залежить від погоди протягом вегетаційного періоду. Наперед її не передбачити, а ймовірностей погодних станів ми не знаємо. Тому культуру обираємо в умовах невизначеності.

Альтернативи:

| Позначення | Культура |
|---|---|
| A1 | Озима пшениця |
| A2 | Кукурудза на зерно |
| A3 | Соняшник |
| A4 | Озимий ріпак |
| A5 | Соя |

Стани природи (погода вегетаційного періоду):

| Позначення | Стан |
|---|---|
| S1 | Сильна посуха |
| S2 | Помірна посуха |
| S3 | Нормальний рік |
| S4 | Вологий рік |

### Функція корисності

За корисність беремо чистий прибуток з одного гектара в тис. грн:

$$u(A_i, S_j) = \frac{Y_{ij}\cdot P_i - C_i}{1000},$$

де $Y_{ij}$ — урожайність культури $i$ за стану $j$ (ц/га), $P_i$ — ціна реалізації (грн/ц), $C_i$ — витрати на вирощування (грн/га). Урожайність, ціни й витрати взято приблизні, для степової зони.

In [1]:
import pandas as pd
from IPython.display import display

alternatives = ["A1 Озима пшениця", "A2 Кукурудза", "A3 Соняшник", "A4 Озимий ріпак", "A5 Соя"]
states = ["S1 Сильна посуха", "S2 Помірна посуха", "S3 Нормальний рік", "S4 Вологий рік"]

yields = pd.DataFrame(
    [[25, 38, 50, 55],
     [30, 60, 100, 130],
     [18, 25, 30, 28],
     [12, 22, 32, 36],
     [10, 20, 30, 35]],
    index=alternatives, columns=states,
)
price = pd.Series([700, 650, 1800, 2000, 1600], index=alternatives)
cost = pd.Series([22_000, 38_000, 26_000, 30_000, 25_000], index=alternatives)

print("Урожайність, ц/га")
display(yields)
print("Ціна та витрати")
display(pd.DataFrame({"Ціна, грн/ц": price, "Витрати, грн/га": cost}))

Урожайність, ц/га


,S1 Сильна посуха,S2 Помірна посуха,S3 Нормальний рік,S4 Вологий рік
A1 Озима пшениця,25,38,50,55
A2 Кукурудза,30,60,100,130
A3 Соняшник,18,25,30,28
A4 Озимий ріпак,12,22,32,36
A5 Соя,10,20,30,35


Ціна та витрати


,"Ціна, грн/ц","Витрати, грн/га"
A1 Озима пшениця,700,22000
A2 Кукурудза,650,38000
A3 Соняшник,1800,26000
A4 Озимий ріпак,2000,30000
A5 Соя,1600,25000


### Матриця рішень

In [2]:
U = yields.mul(price, axis=0).sub(cost, axis=0) / 1000
print("Матриця рішень U, тис. грн/га")
display(U)

Матриця рішень U, тис. грн/га


,S1 Сильна посуха,S2 Помірна посуха,S3 Нормальний рік,S4 Вологий рік
A1 Озима пшениця,-4.5,4.6,13.0,16.5
A2 Кукурудза,-18.5,1.0,27.0,46.5
A3 Соняшник,6.4,19.0,28.0,24.4
A4 Озимий ріпак,-6.0,14.0,34.0,42.0
A5 Соя,-9.0,7.0,23.0,31.0


### Обґрунтування вибору методу

Для вибору використаємо критерій Севіджа (мінімаксного жалю). Ймовірності погодних станів невідомі, а цьому критерію вони й не потрібні. Критерій Лапласа теж обходиться без ймовірностей, але фактично вважає всі стани рівноймовірними, а для погоди таке припущення сумнівне.

Критерій Вальда дивиться лише на найгірший випадок, тобто на сильну посуху, і не враховує, скільки можна недоотримати в нормальний чи вологий рік. Для критерію Гурвіца потрібно задати коефіцієнт песимізму $\alpha$, а обґрунтувати якесь конкретне його значення важко.

Критерій Севіджа показує, скільки прибутку фермер втратить порівняно з культурою, яку він обрав би, якби знав погоду заздалегідь. Для нашої задачі це зрозуміла оцінка, і при цьому враховуються всі стани, а не тільки найгірший.

### Розрахунки за критерієм Севіджа

1. Для кожного стану $S_j$ знаходимо найкращий результат $\beta_j = \max_i u_{ij}$.
2. Будуємо матрицю ризиків (жалю) $r_{ij} = \beta_j - u_{ij}$.
3. Для кожної альтернативи знаходимо максимальний ризик $\max_j r_{ij}$.
4. Обираємо альтернативу з найменшим максимальним ризиком: $\min_i \max_j r_{ij}$.

In [3]:
beta = U.max(axis=0)
print("Крок 1. Найкращий результат для кожного стану β_j, тис. грн/га:")
display(pd.DataFrame({"β_j": beta, "Найкраща альтернатива": U.idxmax(axis=0)}))

R = beta - U
print("Крок 2. Матриця ризиків R, тис. грн/га")
display(R)

max_regret = R.max(axis=1)
print("Крок 3. Максимальний ризик для кожної альтернативи, тис. грн/га")
display(pd.DataFrame({"max r_ij": max_regret, "Стан максимального жалю": R.idxmax(axis=1)}))

choice = max_regret.idxmin()
print(f"Крок 4. min max r = {max_regret.min():.2f} тис. грн/га → {choice}")

Крок 1. Найкращий результат для кожного стану β_j, тис. грн/га:


,β_j,Найкраща альтернатива
S1 Сильна посуха,6.4,A3 Соняшник
S2 Помірна посуха,19.0,A3 Соняшник
S3 Нормальний рік,34.0,A4 Озимий ріпак
S4 Вологий рік,46.5,A2 Кукурудза


Крок 2. Матриця ризиків R, тис. грн/га


,S1 Сильна посуха,S2 Помірна посуха,S3 Нормальний рік,S4 Вологий рік
A1 Озима пшениця,10.9,14.4,21.0,30.0
A2 Кукурудза,24.9,18.0,7.0,0.0
A3 Соняшник,0.0,0.0,6.0,22.1
A4 Озимий ріпак,12.4,5.0,0.0,4.5
A5 Соя,15.4,12.0,11.0,15.5


Крок 3. Максимальний ризик для кожної альтернативи, тис. грн/га


,max r_ij,Стан максимального жалю
A1 Озима пшениця,30.0,S4 Вологий рік
A2 Кукурудза,24.9,S1 Сильна посуха
A3 Соняшник,22.1,S4 Вологий рік
A4 Озимий ріпак,12.4,S1 Сильна посуха
A5 Соя,15.5,S4 Вологий рік


Крок 4. min max r = 12.40 тис. грн/га → A4 Озимий ріпак


### Опис прийнятого рішення

За критерієм Севіджа найкраще сіяти озимий ріпак (A4). Його максимальний жаль становить 12,4 тис. грн/га і виникає за сильної посухи. У решти культур максимальний жаль більший: від 15,5 тис. грн/га у сої до 30 тис. грн/га у пшениці. У нормальний рік ріпак дає найбільший прибуток серед усіх культур (34 тис. грн/га), а за помірної посухи та у вологий рік відстає від найкращого варіанту лише на 5 і 4,5 тис. грн/га.

### Висновки

У лабораторній роботі розв'язано задачу вибору сільськогосподарської культури, коли погода на сезон невідома. Розглянуто 5 культур і 4 погодні стани, за функцію корисності взято чистий прибуток з гектара, побудовано матрицю рішень розміром 5×4.

Рішення приймали за критерієм Севіджа, бо ймовірності станів невідомі, а цей критерій, на відміну від критерію Вальда, враховує втрати в усіх станах і не потребує суб'єктивних параметрів. Оптимальним виявився озимий ріпак: за будь-якої погоди недобір прибутку порівняно з найкращим вибором не перевищить 12,4 тис. грн/га.